# Analyse Exploratoire — Modele CLV Nouvelles Acquisitions

Exploration des donnees propres (`clean_data/*.parquet`) produites par `data_preparation.ipynb`,
structuree autour des **8 hypotheses** du `check_list.md` et de l'architecture a **5 familles de produits**.

> Les colonnes ont ete **standardisees** par `data_preparation.ipynb` : accents retires, snake_case,
> ex. `Revenus EUR avec Liquidity` → `revenus_eur_avec_liquidity`.

## Plan
0. **Fondations** — chargement, panel famille, axe d'anciennete (tenure), cibles cumulees
1. **Paysage des donnees** — cohortes, couverture, profondeur
2. **Population acquise** — demographie, segments, frontaliers
3. **NBI : le moteur de valeur** — distribution, concentration, decomposition par famille, trajectoire
4. **Familles produits** — penetration, encours, zero-inflation
5. **Le snapshot M+3** — l'instant d'entree du modele
6. **H5 — Stabilite de l'equipement dans le temps**
7. **H1 — Pouvoir predictif du NBI precoce**
8. **H2 — Pouvoir predictif des encours M+3**
9. **H7 & churn implicite — Retention et survie**
10. **H3 — Profils de familles differents**
11. **H6 — Taux de croissance par famille**
12. **H8 — Stabilite inter-cohortes**
13. **Apercu de la calibration par deciles**
14. **Synthese pour l'equipe**

> Code en anglais, narration en francais. Graphiques cles exportes dans `../eda_figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

CLEAN_DIR = Path("../clean_data")
FIG_DIR = Path("../eda_figures")
FIG_DIR.mkdir(exist_ok=True)

EPOCH_YEAR = 1994
def idx_to_year(idx):
    return EPOCH_YEAR + (int(idx) - 1) // 12
def idx_to_ym(idx):
    return (EPOCH_YEAR + (int(idx) - 1) // 12, (int(idx) - 1) % 12 + 1)

NBI_COL = "revenus_eur_avec_liquidity"

# Palette par famille (coherente avec la presentation)
FAM_ORDER = ["epargne", "invest", "credit_immo", "autres_cr", "cartes"]
FAM_COLORS = {"epargne": "#06B6D4", "invest": "#7C3AED", "credit_immo": "#EC4899",
              "autres_cr": "#F59E0B", "cartes": "#84CC16"}
FAM_LABELS = {"epargne": "Epargne", "invest": "Investissements", "credit_immo": "Credit immo",
              "autres_cr": "Autres credits", "cartes": "Autre PNB"}

# Mapping encours par famille — EDITABLE (corriger ici si besoin)
FAMILY_ENCOURS = {
    "epargne":      ("savings",   ["avoirs_eur"]),
    "invest":       ("invest",    ["cp_depots_a_terme", "cp_depots_en_cc_epar_liq",
                                   "cp_depots_en_cc_placement", "cp_depots_titres",
                                   "cp_epargne_a_vue"]),
    "invest_banca": ("banca",     ["capitaux_productifs_eur"]),
    "credit_immo":  ("pret_immo", ["credits_utilises_eur"]),
    "autres_cr":    ("credit",    ["credits_utilises_eur"]),
}

plt.rcParams.update({"figure.figsize": (13, 6), "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titlesize": 13, "axes.labelsize": 11, "figure.dpi": 100})
sns.set_palette("muted")

def save_fig(name, fig=None):
    fig = fig or plt.gcf()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=160, bbox_inches="tight", facecolor="white")
    plt.show()

print("Setup OK — figures ->", FIG_DIR.resolve())

## 0. Fondations

Chargement des fichiers propres, construction du **panel famille** (encours + NBI par famille a chaque mois),
de l'axe **anciennete** (`tenure` = mois depuis l'acquisition) et des **cibles cumulees** multi-horizons.

In [ ]:
def load(name):
    p = CLEAN_DIR / f"clean_{name}.parquet"
    if not p.exists():
        print(f"  MANQUANT: {p.name}")
        return None
    df = pd.read_parquet(p)
    print(f"  {name:<12}: {len(df):>10,} lignes x {df.shape[1]} cols")
    return df

print("Chargement:")
stock     = load("stock")
flux      = load("flux")
savings   = load("savings")
invest    = load("invest")
banca     = load("banca")
cartes    = load("cartes")
credit    = load("credit")
pret_immo = load("pret_immo")

PRODUCTS = {"savings": savings, "invest": invest, "banca": banca,
            "cartes": cartes, "credit": credit, "pret_immo": pret_immo}

In [ ]:
# Periode d'acquisition = premier mois observe par client dans STOCK
acq = stock.groupby("identifiant_tiers")["periode"].min().rename("acq_period")
print(f"Clients avec periode d'acquisition: {len(acq):,}")

def add_tenure(df):
    if df is None:
        return None
    df = df.merge(acq, on="identifiant_tiers", how="left")
    df["tenure"] = (df["periode"] - df["acq_period"]).astype("Int64")
    return df

stock = add_tenure(stock)
flux = add_tenure(flux)
for k in list(PRODUCTS):
    PRODUCTS[k] = add_tenure(PRODUCTS[k])
savings, invest, banca, cartes, credit, pret_immo = (PRODUCTS[k] for k in
    ["savings", "invest", "banca", "cartes", "credit", "pret_immo"])

# NBI column alias on FLUX
if NBI_COL in flux.columns:
    flux["nbi"] = flux[NBI_COL]

print("\nDistribution de l'anciennete (FLUX):")
print(flux["tenure"].describe().round(1).to_string())

In [ ]:
# ── Panel famille : encours + NBI agreges a (client, periode) par famille ────
def agg_encours(df, cols):
    if df is None:
        return None
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return None
    tmp = df[["identifiant_tiers", "periode"]].copy()
    tmp["encours"] = df[cols].sum(axis=1, min_count=1)
    return tmp.groupby(["identifiant_tiers", "periode"], as_index=False)["encours"].sum()

def agg_nbi(df):
    if df is None or NBI_COL not in df.columns:
        return None
    tmp = df[["identifiant_tiers", "periode", NBI_COL]].rename(columns={NBI_COL: "nbi"})
    return tmp.groupby(["identifiant_tiers", "periode"], as_index=False)["nbi"].sum()

def combine(a, b, val):
    parts = [x for x in (a, b) if x is not None]
    if not parts:
        return None
    return pd.concat(parts).groupby(["identifiant_tiers", "periode"], as_index=False)[val].sum()

# Encours par famille (Investissements = INVEST + BANCA)
enc = {
    "epargne":     agg_encours(savings, FAMILY_ENCOURS["epargne"][1]),
    "invest":      combine(agg_encours(invest, FAMILY_ENCOURS["invest"][1]),
                           agg_encours(banca, FAMILY_ENCOURS["invest_banca"][1]), "encours"),
    "credit_immo": agg_encours(pret_immo, FAMILY_ENCOURS["credit_immo"][1]),
    "autres_cr":   agg_encours(credit, FAMILY_ENCOURS["autres_cr"][1]),
}
# NBI par famille
nbi = {
    "epargne":     agg_nbi(savings),
    "invest":      combine(agg_nbi(invest), agg_nbi(banca), "nbi"),
    "credit_immo": agg_nbi(pret_immo),
    "autres_cr":   agg_nbi(credit),
    "cartes":      agg_nbi(cartes),
}

# Panel long unifie : family, client, periode, tenure, encours, nbi
parts = []
for fam in FAM_ORDER:
    n = nbi.get(fam)
    e = enc.get(fam)
    if n is None and e is None:
        continue
    base = n.copy() if n is not None else e[["identifiant_tiers", "periode"]].copy()
    if n is None:
        base["nbi"] = np.nan
    if e is not None:
        base = base.merge(e, on=["identifiant_tiers", "periode"], how="outer")
    else:
        base["encours"] = np.nan
    base["family"] = fam
    parts.append(base)

panel = pd.concat(parts, ignore_index=True)
panel = panel.merge(acq, on="identifiant_tiers", how="left")
panel = panel.dropna(subset=["acq_period"])
panel["tenure"] = (panel["periode"] - panel["acq_period"]).astype(int)
panel["cohort"] = panel["acq_period"].map(idx_to_year)
panel = panel[panel["tenure"] >= 0]

print(f"Panel famille: {len(panel):,} lignes (client x periode x famille)")
print(panel.groupby("family").agg(lignes=("nbi", "size"),
        nbi_moyen=("nbi", "mean"), encours_moyen=("encours", "mean")).round(1).to_string())

In [ ]:
# ── Outils reutilisables : eligibilite, cumul NBI, snapshot ──────────────────
max_tenure = flux.groupby("identifiant_tiers")["tenure"].max()
cohort_of = acq.map(idx_to_year)

def eligible_clients(horizon):
    """Clients dont la fenetre permet d'observer `horizon` mois."""
    return set(max_tenure[max_tenure >= horizon - 1].index)

def cum_nbi_client(horizon, fam=None):
    """NBI cumule par client sur les `horizon` premiers mois (tenure 0..horizon-1)."""
    p = panel if fam is None else panel[panel["family"] == fam]
    p = p[p["tenure"] < horizon]
    return p.groupby("identifiant_tiers")["nbi"].sum()

def value_at_tenure(values, fam, t):
    """Encours d'une famille a un tenure donne, par client."""
    p = panel[(panel["family"] == fam) & (panel["tenure"] == t)]
    return p.groupby("identifiant_tiers")[values].sum()

HORIZONS = [12, 24, 36, 48, 60]
print("Clients eligibles par horizon:")
for h in HORIZONS:
    print(f"  M+{h}: {len(eligible_clients(h)):,} clients")

# Recap general
n_clients = stock["identifiant_tiers"].nunique()
print(f"\nRECAP — {n_clients:,} clients | "
      f"anciennete mediane {int(max_tenure.median())} mois | "
      f"max {int(max_tenure.max())} mois")

## 1. Paysage des donnees

Combien de clients par cohorte d'acquisition, et jusqu'ou peut-on les suivre ?
L'**entonnoir de couverture** indique quelles cohortes alimentent les analyses long-terme :
seules les cohortes anciennes (2018-2020) atteignent M+60.

In [ ]:
clients_df = pd.DataFrame({"cohort": cohort_of}).join(max_tenure.rename("max_tenure"))
vol = clients_df["cohort"].value_counts().sort_index()
milestones = [3, 12, 24, 36, 48, 60]
cov = {m: (clients_df["max_tenure"] >= m).mean() * 100 for m in milestones}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(vol.index.astype(int).astype(str), vol.values, color="#334155", edgecolor="white")
axes[0].set_title("Clients eligibles par cohorte d'acquisition")
axes[0].set_ylabel("Nombre de clients")
for i, v in enumerate(vol.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

bars = axes[1].bar([f"M+{m}" for m in milestones], [cov[m] for m in milestones],
                   color="#7C3AED", edgecolor="white")
axes[1].set_title("Entonnoir de couverture")
axes[1].set_ylabel("% clients atteignant cette anciennete")
axes[1].set_ylim(0, 105)
for i, m in enumerate(milestones):
    axes[1].text(i, cov[m] + 1.5, f"{cov[m]:.0f}%", ha="center", fontsize=10)
plt.tight_layout()
save_fig("01_cohorts_coverage")

# Profondeur par cohorte
depth = clients_df.groupby("cohort")["max_tenure"].agg(["count", "median", "max"]).round(0)
print("Profondeur d'historique par cohorte (mois):")
print(depth.to_string())

## 2. Population acquise

Qui sont les nouveaux clients ? Demographie, segments et — dimension cle au Luxembourg —
la part de **frontaliers**. Le snapshot le plus recent par client sert de reference.

In [ ]:
latest = stock.sort_values("periode").groupby("identifiant_tiers").last()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
if "age" in latest.columns:
    axes[0].hist(pd.to_numeric(latest["age"], errors="coerce").dropna().clip(18, 90),
                 bins=40, color="#06B6D4", edgecolor="white")
    axes[0].set_title("Distribution de l'age")
    axes[0].set_xlabel("Age")
if "revenu_total" in latest.columns:
    rev = pd.to_numeric(latest["revenu_total"], errors="coerce").dropna()
    rev = rev[rev > 0]
    if len(rev):
        axes[1].hist(rev.clip(upper=rev.quantile(0.98)), bins=50, color="#84CC16", edgecolor="white")
    axes[1].set_title("Distribution du revenu total")
    axes[1].set_xlabel("EUR")
if "nombre_denfants" in latest.columns:
    pd.to_numeric(latest["nombre_denfants"], errors="coerce").clip(0, 6).value_counts().sort_index().plot(
        kind="bar", ax=axes[2], color="#F59E0B", edgecolor="white")
    axes[2].set_title("Nombre d'enfants")
    plt.sca(axes[2]); plt.xticks(rotation=0)
plt.tight_layout()
save_fig("02_demographics")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
seg_col = next((c for c in ["segment_ameliore", "segment_upgrade"] if c in latest.columns), None)
if seg_col:
    latest[seg_col].astype(str).value_counts().head(10).plot(
        kind="barh", ax=axes[0], color="#7C3AED", edgecolor="white")
    axes[0].set_title(f"Segments ({seg_col})")
if "frontalier_o_n" in latest.columns:
    fr = (latest["frontalier_o_n"].astype(str).str.strip().str.upper()
          .map({"O": "Frontalier", "N": "Resident / Autre"}).value_counts())
    axes[1].pie(fr.values, labels=fr.index, autopct="%1.0f%%",
                colors=["#EC4899", "#06B6D4"], startangle=90)
    axes[1].set_title("Frontaliers vs Residents")
if "type_de_profession" in latest.columns:
    latest["type_de_profession"].astype(str).value_counts().head(10).plot(
        kind="barh", ax=axes[2], color="#334155", edgecolor="white")
    axes[2].set_title("Professions")
plt.tight_layout()
save_fig("03_population")

## 3. NBI : le moteur de valeur

La cible ultime du modele est le NBI. On regarde :
- sa **distribution** (longue traine typique du revenu bancaire),
- sa **concentration** (courbe de Lorenz — quel % de clients fait quel % du NBI),
- sa **decomposition par famille** (quelle famille genere le plus de valeur),
- sa **trajectoire** sur l'axe d'anciennete (croissance, plateau ou declin).

In [ ]:
# NBI total cumule par client (depuis FLUX)
client_nbi = flux.groupby("identifiant_tiers")["nbi"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
pos = client_nbi[client_nbi > 0]
axes[0].hist(pos.clip(upper=pos.quantile(0.98)), bins=70, color="#334155", edgecolor="white")
axes[0].axvline(pos.median(), color="#EC4899", ls="--", label=f"Mediane = {pos.median():,.0f}")
axes[0].axvline(pos.mean(), color="#F59E0B", ls="--", label=f"Moyenne = {pos.mean():,.0f}")
axes[0].set_title("Distribution du NBI cumule par client (>0, clip 98%)")
axes[0].set_xlabel("EUR"); axes[0].legend()

# Lorenz
v = pos.sort_values()
cum = np.cumsum(v.values) / v.sum() * 100
xx = np.arange(1, len(v) + 1) / len(v) * 100
axes[1].plot(xx, cum, lw=2.5, color="#7C3AED", label="Courbe de Lorenz")
axes[1].plot([0, 100], [0, 100], "--", color="gray", alpha=0.6, label="Egalite parfaite")
axes[1].set_xlabel("% des clients (NBI croissant)")
axes[1].set_ylabel("% du NBI total")
axes[1].set_title("Concentration du NBI"); axes[1].legend()
plt.tight_layout()
save_fig("04_nbi_distribution_lorenz")

vd = pos.sort_values(ascending=False)
print("Concentration du NBI (analyse de Pareto):")
for pct in [1, 5, 10, 20, 50]:
    k = max(int(len(vd) * pct / 100), 1)
    print(f"  Top {pct:2d}% des clients ({k:,}) -> {vd.head(k).sum() / vd.sum() * 100:.1f}% du NBI")
print(f"\n  NBI median: {pos.median():,.0f} EUR | moyen: {pos.mean():,.0f} EUR | "
      f"% clients NBI<=0: {(client_nbi <= 0).mean() * 100:.1f}%")

In [ ]:
# Decomposition du NBI par famille
fam_nbi = pd.Series({fam: panel[panel["family"] == fam]["nbi"].sum() for fam in FAM_ORDER})
fam_nbi = fam_nbi[fam_nbi > 0]
colors = [FAM_COLORS[f] for f in fam_nbi.index]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar([FAM_LABELS[f] for f in fam_nbi.index], fam_nbi.values / 1e6, color=colors, edgecolor="white")
axes[0].set_title("NBI total par famille de produit")
axes[0].set_ylabel("NBI total (M EUR)")
plt.sca(axes[0]); plt.xticks(rotation=20, ha="right")
for i, v in enumerate(fam_nbi.values):
    axes[0].text(i, v / 1e6, f"{v/1e6:.1f}M", ha="center", va="bottom", fontsize=9)

axes[1].pie(fam_nbi.values, labels=[FAM_LABELS[f] for f in fam_nbi.index],
            autopct="%1.0f%%", colors=colors, startangle=90)
axes[1].set_title("Part de chaque famille dans le NBI total")
plt.tight_layout()
save_fig("05_nbi_by_family")

print("Contribution au NBI total:")
for f in fam_nbi.sort_values(ascending=False).index:
    print(f"  {FAM_LABELS[f]:<18}: {fam_nbi[f]/fam_nbi.sum()*100:5.1f}%")

In [ ]:
# Trajectoire du NBI mensuel moyen sur l'axe d'anciennete
TMAX = 60
fig, ax = plt.subplots(figsize=(13, 6))
total_traj = flux[flux["tenure"].between(0, TMAX)].groupby("tenure")["nbi"].mean()
ax.plot(total_traj.index, total_traj.values, lw=3, color="#334155", label="Total (FLUX)")
for fam in FAM_ORDER:
    tr = panel[(panel["family"] == fam) & (panel["tenure"].between(0, TMAX))].groupby("tenure")["nbi"].mean()
    if len(tr):
        ax.plot(tr.index, tr.values, lw=1.8, color=FAM_COLORS[fam], label=FAM_LABELS[fam], alpha=0.9)
ax.set_xlabel("Mois depuis l'acquisition (tenure)")
ax.set_ylabel("NBI mensuel moyen (EUR)")
ax.set_title("Trajectoire du NBI mensuel moyen par famille")
ax.axvline(3, color="red", ls=":", alpha=0.6, label="Snapshot M+3")
ax.legend(ncol=2)
plt.tight_layout()
save_fig("06_nbi_trajectory")

## 4. Familles produits — penetration & encours

Pour chaque famille : quelle proportion de clients la detient, quel encours moyen,
et surtout le **taux de zero (H4)**. Le credit immobilier est attendu tres creux —
peu de nouveaux clients ont un pret immo a M+3. Une famille trop creuse justifie un
**modele en deux etapes** (classifieur zero/non-zero + regresseur).

In [ ]:
rows = []
for fam in ["epargne", "invest", "credit_immo", "autres_cr"]:
    sub = panel[panel["family"] == fam]
    if not len(sub):
        continue
    holders = sub[sub["encours"].fillna(0) > 0]["identifiant_tiers"].nunique()
    pen = holders / max(len(eligible_clients(3)), 1) * 100
    enc_pos = sub[sub["encours"].fillna(0) > 0]["encours"]
    # zero-inflation a M+3
    m3 = sub[sub["tenure"] == 3]
    zero_m3 = (m3["encours"].fillna(0) <= 0).mean() * 100 if len(m3) else np.nan
    rows.append({"famille": FAM_LABELS[fam], "detenteurs": holders,
                 "penetration_%": round(pen, 1),
                 "encours_median": round(enc_pos.median(), 0) if len(enc_pos) else 0,
                 "encours_moyen": round(enc_pos.mean(), 0) if len(enc_pos) else 0,
                 "zero_a_M+3_%": round(zero_m3, 1)})
pen_df = pd.DataFrame(rows)
print(pen_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fams = [f for f in ["epargne", "invest", "credit_immo", "autres_cr"]]
cols = [FAM_COLORS[f] for f in fams]
axes[0].bar(pen_df["famille"], pen_df["penetration_%"], color=cols, edgecolor="white")
axes[0].set_title("Taux de penetration par famille"); axes[0].set_ylabel("% clients detenteurs")
plt.sca(axes[0]); plt.xticks(rotation=20, ha="right")
axes[1].bar(pen_df["famille"], pen_df["zero_a_M+3_%"], color=cols, edgecolor="white")
axes[1].set_title("Zero-inflation a M+3 (H4)"); axes[1].set_ylabel("% clients avec encours = 0")
axes[1].axhline(60, color="red", ls="--", alpha=0.6, label="Seuil modele 2 etapes (60%)")
axes[1].legend(); plt.sca(axes[1]); plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_fig("07_penetration_zero")

In [ ]:
# Trajectoire d'encours moyen par famille (detenteurs uniquement)
TMAX = 60
fig, ax = plt.subplots(figsize=(13, 6))
for fam in ["epargne", "invest", "credit_immo", "autres_cr"]:
    sub = panel[(panel["family"] == fam) & (panel["tenure"].between(0, TMAX)) & (panel["encours"].fillna(0) > 0)]
    tr = sub.groupby("tenure")["encours"].mean()
    if len(tr):
        ax.plot(tr.index, tr.values, lw=2, color=FAM_COLORS[fam], label=FAM_LABELS[fam])
ax.set_xlabel("Mois depuis l'acquisition")
ax.set_ylabel("Encours moyen des detenteurs (EUR)")
ax.set_title("Trajectoire d'encours par famille (detenteurs)")
ax.axvline(3, color="red", ls=":", alpha=0.6, label="M+3")
ax.legend()
plt.tight_layout()
save_fig("08_encours_trajectory")

## 5. Le snapshot M+3 — l'instant d'entree du modele

Le modele observe chaque client exactement **3 mois apres l'acquisition**.
On construit ici le tableau `snap3` (encours par famille, nombre de produits, NBI precoce)
reutilise dans les sections suivantes.

In [ ]:
# Encours par famille a M+3 (pivot client x famille)
m3 = panel[panel["tenure"] == 3]
snap3 = m3.pivot_table(index="identifiant_tiers", columns="family",
                       values="encours", aggfunc="sum")
snap3.columns = [f"enc_{c}_m3" for c in snap3.columns]

# Nombre de produits detenus a M+3 (familles avec encours>0 ou nbi>0)
held = m3[(m3["encours"].fillna(0) > 0) | (m3["nbi"].fillna(0) > 0)]
snap3["n_produits_m3"] = held.groupby("identifiant_tiers")["family"].nunique()
snap3["n_produits_m3"] = snap3["n_produits_m3"].fillna(0).astype(int)

# NBI des 3 premiers mois (tenure 0,1,2) — total et par famille
early = panel[panel["tenure"] < 3]
snap3["nbi_3m"] = early.groupby("identifiant_tiers")["nbi"].sum()
snap3 = snap3.join(cohort_of.rename("cohort"))
if seg_col:
    snap3 = snap3.join(latest[seg_col].rename("segment"))

print(f"snap3: {len(snap3):,} clients observes a M+3")
print(f"\nNombre de produits a M+3:")
print(snap3["n_produits_m3"].value_counts().sort_index().to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
snap3["n_produits_m3"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color="#7C3AED", edgecolor="white")
axes[0].set_title("Nombre de produits detenus a M+3"); axes[0].set_xlabel("Nb produits")
plt.sca(axes[0]); plt.xticks(rotation=0)
nbi3 = snap3["nbi_3m"].dropna()
axes[1].hist(nbi3.clip(nbi3.quantile(0.02), nbi3.quantile(0.98)), bins=60,
             color="#06B6D4", edgecolor="white")
axes[1].set_title("NBI des 3 premiers mois"); axes[1].set_xlabel("EUR")
plt.tight_layout()
save_fig("09_snapshot_m3")

## 6. H5 — Stabilite de l'equipement dans le temps

**Question :** l'equipement observe a M+3 represente-t-il bien l'equipement futur ?
Si les clients acquierent l'essentiel de leurs produits *apres* M+3, le snapshot les rate.
On compare le nombre de produits detenus a M+3, M+12, M+36 et M+60 (cohortes a historique complet),
et on mesure le ratio M+3 / M+horizon.

**Si valide** (ratio eleve) : le snapshot M+3 est fiable.
**Sinon** : les facteurs de croissance devront compenser l'enrichissement futur.

In [ ]:
def n_products_at(t):
    s = panel[(panel["tenure"] == t) & ((panel["encours"].fillna(0) > 0) | (panel["nbi"].fillna(0) > 0))]
    return s.groupby("identifiant_tiers")["family"].nunique()

milestones = [3, 12, 24, 36, 48, 60]
np_by_t = {t: n_products_at(t) for t in milestones}

# Moyenne sur les clients presents a chaque jalon
mean_prod = {t: np_by_t[t].mean() for t in milestones}
ratio_vs_m3 = {t: mean_prod[3] / mean_prod[t] * 100 for t in milestones if mean_prod[t] > 0}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(list(mean_prod.keys()), list(mean_prod.values()), "o-", lw=2.5,
             color="#7C3AED", markersize=9)
axes[0].set_title("Nombre moyen de produits selon l'anciennete")
axes[0].set_xlabel("Mois depuis l'acquisition"); axes[0].set_ylabel("Nb produits moyen")
for t in milestones:
    axes[0].text(t, mean_prod[t], f"{mean_prod[t]:.2f}", ha="center", va="bottom", fontsize=9)

xs = [t for t in milestones if t in ratio_vs_m3]
axes[1].bar([f"M+{t}" for t in xs], [ratio_vs_m3[t] for t in xs], color="#06B6D4", edgecolor="white")
axes[1].set_title("Equipement M+3 en % de l'equipement a M+horizon")
axes[1].set_ylabel("% (M+3 / M+horizon)"); axes[1].set_ylim(0, 110)
for i, t in enumerate(xs):
    axes[1].text(i, ratio_vs_m3[t] + 1.5, f"{ratio_vs_m3[t]:.0f}%", ha="center", fontsize=10)
plt.tight_layout()
save_fig("10_H5_equipment_stability")

print("Equipement a M+3 vs horizons futurs:")
for t in xs:
    print(f"  M+3 represente {ratio_vs_m3[t]:.0f}% des produits detenus a M+{t}")

## 7. H1 — Pouvoir predictif du NBI precoce

Le NBI observe durant les **3 premiers mois** est-il correle au NBI cumule a long terme ?
On mesure la correlation (Spearman, robuste a la longue traine) entre `nbi_3m` et le NBI cumule
a 12 / 24 / 36 mois, et on trace la **courbe de correlation progressive** : combien de mois
d'observation faut-il pour bien predire ?

**Si valide** : `nbi_3m` est une feature cle. **Sinon** : s'appuyer sur les encours et l'equipement.

In [ ]:
# Correlation NBI precoce (3 mois) vs NBI cumule a divers horizons
print("Correlation (Spearman) entre NBI des 3 premiers mois et NBI cumule:")
res = []
for h in [12, 24, 36]:
    elig = eligible_clients(h)
    cum_h = cum_nbi_client(h)
    s = snap3["nbi_3m"]
    common = list(set(s.dropna().index) & set(cum_h.index) & elig)
    if len(common) > 30:
        rho, _ = stats.spearmanr(s.loc[common], cum_h.loc[common])
        r, _ = stats.pearsonr(s.loc[common].fillna(0), cum_h.loc[common].fillna(0))
        res.append({"horizon": f"M+{h}", "n": len(common),
                    "spearman": round(rho, 3), "pearson": round(r, 3)})
print(pd.DataFrame(res).to_string(index=False))

# Courbe progressive : k premiers mois vs NBI cumule a M+36
H = 36
elig = eligible_clients(H)
cum_target = cum_nbi_client(H)
prog = []
for k in range(1, 13):
    early_k = panel[panel["tenure"] < k].groupby("identifiant_tiers")["nbi"].sum()
    common = list(set(early_k.index) & set(cum_target.index) & elig)
    if len(common) > 30:
        rho, _ = stats.spearmanr(early_k.loc[common], cum_target.loc[common])
        prog.append((k, rho))

fig, ax = plt.subplots(figsize=(11, 6))
if prog:
    ks, rhos = zip(*prog)
    ax.plot(ks, rhos, "o-", lw=2.5, color="#7C3AED", markersize=8)
    ax.axvline(3, color="red", ls=":", alpha=0.7, label="Snapshot M+3")
    ax.axhline(0.7, color="green", ls="--", alpha=0.5, label="Fort (0.7)")
    ax.axhline(0.5, color="orange", ls="--", alpha=0.5, label="Modere (0.5)")
ax.set_xlabel("Nombre de mois de NBI cumules en observation")
ax.set_ylabel(f"Correlation Spearman avec NBI cumule M+{H}")
ax.set_title("H1 — Correlation progressive du NBI precoce")
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()
save_fig("11_H1_early_nbi")

## 8. H2 — Pouvoir predictif des encours M+3

L'encours observe a M+3 est-il un bon predicteur du NBI futur de la **meme famille** ?
C'est le pilier du modele : si l'encours initial porte le signal, chaque sous-modele a une base solide.

In [ ]:
print("Correlation (Spearman) encours M+3 -> NBI cumule M+36 de la meme famille:")
H = 36
elig = eligible_clients(H)
res = []
for fam in ["epargne", "invest", "credit_immo", "autres_cr"]:
    col = f"enc_{fam}_m3"
    if col not in snap3.columns:
        continue
    enc_m3 = snap3[col].dropna()
    enc_m3 = enc_m3[enc_m3 > 0]                      # detenteurs a M+3
    cum_h = cum_nbi_client(H, fam=fam)
    common = list(set(enc_m3.index) & set(cum_h.index) & elig)
    if len(common) > 30:
        rho, _ = stats.spearmanr(enc_m3.loc[common], cum_h.loc[common])
        res.append({"famille": FAM_LABELS[fam], "n_detenteurs": len(common),
                    "spearman": round(rho, 3),
                    "nbi_moyen_M36": round(cum_h.loc[common].mean(), 0)})
res_df = pd.DataFrame(res)
print(res_df.to_string(index=False))

if len(res_df):
    fig, ax = plt.subplots(figsize=(11, 5))
    cols = [FAM_COLORS[f] for f in ["epargne", "invest", "credit_immo", "autres_cr"]
            if FAM_LABELS[f] in res_df["famille"].values]
    ax.bar(res_df["famille"], res_df["spearman"], color=cols, edgecolor="white")
    ax.axhline(0.5, color="orange", ls="--", alpha=0.6, label="Modere")
    ax.axhline(0.7, color="green", ls="--", alpha=0.6, label="Fort")
    ax.set_title("H2 — Correlation encours M+3 vs NBI futur (par famille)")
    ax.set_ylabel("Spearman"); ax.set_ylim(0, 1); ax.legend()
    plt.tight_layout()
    save_fig("12_H2_encours_m3")

## 9. H7 & churn implicite — Retention et survie

Le modele n'a pas de sous-modele de churn : l'attrition est **implicitement capturee** par les encours
nuls apres le depart. On valide cela par une analyse de survie : quelle fraction de clients reste
**active** (NBI>0 ou encours>0) a M+12, M+24, ... M+60 ? Et les clients **multi-equipes a M+3**
restent-ils plus longtemps ?

In [ ]:
# Dernier mois actif par client
active = panel[(panel["nbi"].fillna(0) > 0) | (panel["encours"].fillna(0) > 0)]
last_active = active.groupby("identifiant_tiers")["tenure"].max()
surv = pd.DataFrame({"max_obs": max_tenure, "last_active": last_active}).dropna()

ms = [12, 24, 36, 48, 60]
rates = []
for t in ms:
    elig = surv[surv["max_obs"] >= t]
    rate = (elig["last_active"] >= t).mean() * 100 if len(elig) else np.nan
    rates.append(rate)

# Retention a M+36 selon le nombre de produits a M+3
surv2 = surv.join(snap3["n_produits_m3"])
elig36 = surv2[surv2["max_obs"] >= 36].copy()
elig36["bucket"] = elig36["n_produits_m3"].clip(0, 4).map({0: "0-1", 1: "0-1", 2: "2", 3: "3", 4: "4+"})
ret_by_prod = elig36.groupby("bucket").apply(lambda g: (g["last_active"] >= 36).mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot([f"M+{t}" for t in ms], rates, "o-", lw=2.5, color="#EC4899", markersize=9)
axes[0].set_title("Courbe de survie (clients encore actifs)")
axes[0].set_ylabel("% clients actifs"); axes[0].set_ylim(0, 105)
for i, r in enumerate(rates):
    if pd.notna(r):
        axes[0].text(i, r + 1.5, f"{r:.0f}%", ha="center", fontsize=10)
if len(ret_by_prod):
    axes[1].bar(ret_by_prod.index.astype(str), ret_by_prod.values, color="#84CC16", edgecolor="white")
    axes[1].set_title("Retention a M+36 selon l'equipement a M+3 (H7)")
    axes[1].set_xlabel("Nb produits a M+3"); axes[1].set_ylabel("% actifs a M+36"); axes[1].set_ylim(0, 105)
plt.tight_layout()
save_fig("13_H7_survival")

print("Taux de survie:")
for t, r in zip(ms, rates):
    print(f"  Actifs a M+{t}: {r:.1f}%")

## 10. H3 — Profils de familles differents

Si tous les clients avaient la meme repartition d'encours, 5 sous-modeles seraient redondants.
On verifie l'**heterogeneite** du mix produit et les **correlations inter-familles**
(ex : epargne et investissements vont-ils de pair ? credit et immobilier ?).

In [ ]:
# Matrice de correlation des encours M+3 entre familles
enc_cols = [f"enc_{f}_m3" for f in ["epargne", "invest", "credit_immo", "autres_cr"] if f"enc_{f}_m3" in snap3.columns]
corr = snap3[enc_cols].fillna(0).corr(method="spearman")
corr.index = [c.replace("enc_", "").replace("_m3", "") for c in corr.index]
corr.columns = corr.index

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=axes[0], linewidths=0.5)
axes[0].set_title("Correlation inter-familles (encours M+3)")

# Part de chaque famille dans le NBI du client (heterogeneite)
fam_nbi_client = (panel.groupby(["identifiant_tiers", "family"])["nbi"].sum().unstack(fill_value=0))
fam_nbi_client = fam_nbi_client[fam_nbi_client.sum(axis=1) > 0]
shares = fam_nbi_client.div(fam_nbi_client.sum(axis=1), axis=0)
mean_share = shares.mean().reindex(FAM_ORDER).dropna()
axes[1].bar([FAM_LABELS[f] for f in mean_share.index], mean_share.values * 100,
            color=[FAM_COLORS[f] for f in mean_share.index], edgecolor="white")
axes[1].set_title("Part moyenne de chaque famille dans le NBI client")
axes[1].set_ylabel("% du NBI client"); plt.sca(axes[1]); plt.xticks(rotation=20, ha="right")
plt.tight_layout()
save_fig("14_H3_family_profiles")

## 11. H6 — Taux de croissance par famille

La CLV projette le NBI sur 5 ans avec un facteur de croissance `g`. On estime ici la
**croissance annuelle de l'encours** par famille (annee 1 -> 2 -> 3 -> 4).
Une croissance stable = facteur `g` constant ; instable = facteurs conditionnels.

In [ ]:
# Encours moyen par annee d'anciennete et par famille
def year_bucket(t):
    return t // 12 + 1   # annee 1 = mois 0-11

panel["annee_anc"] = panel["tenure"].map(year_bucket)
growth_rows = []
fig, ax = plt.subplots(figsize=(12, 6))
for fam in ["epargne", "invest", "credit_immo", "autres_cr"]:
    sub = panel[(panel["family"] == fam) & (panel["encours"].fillna(0) > 0) & (panel["annee_anc"].between(1, 5))]
    by_year = sub.groupby("annee_anc")["encours"].mean()
    if len(by_year) >= 2:
        ax.plot(by_year.index, by_year.values, "o-", lw=2, color=FAM_COLORS[fam], label=FAM_LABELS[fam])
        for y in by_year.index[1:]:
            if y - 1 in by_year.index and by_year[y - 1] > 0:
                g = by_year[y] / by_year[y - 1] - 1
                growth_rows.append({"famille": FAM_LABELS[fam], "annee": f"{y-1}->{y}",
                                    "croissance_%": round(g * 100, 1)})
ax.set_xlabel("Annee d'anciennete"); ax.set_ylabel("Encours moyen (detenteurs, EUR)")
ax.set_title("H6 — Evolution annuelle de l'encours par famille"); ax.legend()
plt.tight_layout()
save_fig("15_H6_growth")

print("Taux de croissance annuels de l'encours:")
print(pd.DataFrame(growth_rows).pivot_table(index="famille", columns="annee",
      values="croissance_%", aggfunc="first").to_string())

## 12. H8 — Stabilite inter-cohortes

La relation entre comportement precoce et valeur future est-elle stable entre cohortes
(2018 vs 2021 vs 2024) ? Une derive imposerait un re-entrainement frequent.
On compare le NBI precoce moyen et la correlation `nbi_3m -> NBI M+12` par cohorte.

In [ ]:
H = 12
elig = eligible_clients(H)
cum12 = cum_nbi_client(H)
rows = []
for c in sorted(snap3["cohort"].dropna().unique()):
    idx = snap3[snap3["cohort"] == c].index
    common = list(set(idx) & set(cum12.index) & elig)
    if len(common) > 50:
        s = snap3.loc[common, "nbi_3m"]
        rho, _ = stats.spearmanr(s, cum12.loc[common])
        rows.append({"cohort": int(c), "n": len(common),
                     "nbi_3m_moyen": round(s.mean(), 0),
                     "nbi_M12_moyen": round(cum12.loc[common].mean(), 0),
                     "spearman_3m_vs_M12": round(rho, 3)})
drift = pd.DataFrame(rows)
print(drift.to_string(index=False))

if len(drift):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(drift["cohort"].astype(str), drift["nbi_3m_moyen"], color="#06B6D4", edgecolor="white")
    axes[0].set_title("NBI moyen des 3 premiers mois par cohorte"); axes[0].set_ylabel("EUR")
    axes[1].plot(drift["cohort"].astype(str), drift["spearman_3m_vs_M12"], "o-", lw=2.5,
                 color="#7C3AED", markersize=9)
    axes[1].set_title("Stabilite de la correlation (nbi_3m -> NBI M+12)")
    axes[1].set_ylabel("Spearman"); axes[1].set_ylim(0, 1)
    plt.tight_layout()
    save_fig("16_H8_cohort_stability")

## 13. Apercu de la calibration par deciles

Demonstration de la mecanique du modele : on classe les clients par **encours d'epargne a M+3**,
on forme 10 deciles, et on affiche le **NBI cumule observe moyen** par decile. Une relation
monotone (croissante) confirme que le decile est un bon resume — et que remplacer la prediction
brute par la moyenne du decile lisse les outliers.

In [ ]:
H = 36
elig = eligible_clients(H)
proxy = snap3["enc_epargne_m3"].dropna()
proxy = proxy[proxy > 0]
target = cum_nbi_client(H)  # NBI total cumule M+36
common = list(set(proxy.index) & set(target.index) & elig)

if len(common) > 100:
    d = pd.DataFrame({"proxy": proxy.loc[common], "target": target.loc[common]})
    d["decile"] = pd.qcut(d["proxy"].rank(method="first"), 10, labels=range(1, 11))
    dec_stats = d.groupby("decile").agg(encours_moyen=("proxy", "mean"),
                                        nbi_M36_moyen=("target", "mean"),
                                        n=("target", "size"))
    print(dec_stats.round(0).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(dec_stats.index.astype(str), dec_stats["encours_moyen"], color="#06B6D4", edgecolor="white")
    axes[0].set_title("Encours epargne moyen par decile (M+3)")
    axes[0].set_xlabel("Decile"); axes[0].set_ylabel("Encours moyen (EUR)")
    axes[1].bar(dec_stats.index.astype(str), dec_stats["nbi_M36_moyen"], color="#7C3AED", edgecolor="white")
    axes[1].set_title("NBI cumule observe M+36 par decile -> valeur calibree")
    axes[1].set_xlabel("Decile"); axes[1].set_ylabel("NBI cumule moyen (EUR)")
    plt.tight_layout()
    save_fig("17_decile_calibration")
else:
    print("Pas assez de clients eligibles pour la demo deciles.")

## 14. Synthese pour l'equipe

**A completer apres execution** — chaque hypothese recoit un verdict (validee / a creuser / invalidee)
et le chiffre cle correspondant.

| Hyp. | Question | Chiffre cle | Verdict |
|------|----------|-------------|---------|
| H1 | NBI precoce predictif ? | Spearman(nbi_3m, NBI M+36) = ___ | |
| H2 | Encours M+3 predictif ? | Spearman par famille = ___ | |
| H3 | Familles differentes ? | Correlations inter-familles = ___ | |
| H4 | Assez de non-zeros ? | Zero a M+3 : immo ___% , autres ___% | |
| H5 | Equipement stable ? | M+3 = ___% de M+60 | |
| H6 | Croissance stable ? | Croissance annuelle = ___% | |
| H7 | Multi-equipes restent ? | Retention M+36 : 4+ produits vs 0-1 = ___ | |
| H8 | Stable entre cohortes ? | Spearman 2018 vs 2024 = ___ | |

### Implications pour le design
- **Familles creuses (H4)** : credit immo necessite-t-il un modele 2 etapes ?
- **Snapshot M+3 (H5)** : suffisant, ou faut-il renforcer les facteurs de croissance ?
- **Feature NBI precoce (H1)** : a inclure dans les sous-modeles ?
- **Stabilite (H8)** : frequence de re-entrainement (annuelle vs trimestrielle).

> Espace libre pour les analyses complementaires de l'equipe.